# Gaming Laptops Vizualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('laptops_cleaned.csv')

print(df.head().to_string())

In [ ]:
# Clean the brand column
df["laptop_brand"] = df["laptop_brand"].astype(str).str.strip()

# Calculate counts and percentages
brand_counts = df["laptop_brand"].value_counts()
brand_pct = (brand_counts / brand_counts.sum()) * 100

# Sort the data in ascending order for barh (so the largest is at the top)
brand_pct_sorted = brand_pct.sort_values(ascending=True)

# Identify the top brand
top_brand = brand_counts.idxmax()

# Create the plot
fig, ax = plt.subplots(figsize=(10, 8))

# Color the top brand red and the rest light blue
colors = ["#FF4C4C" if brand == top_brand else "#A9C7E8" for brand in brand_pct_sorted.index]

# Create horizontal bars
bars = ax.barh(brand_pct_sorted.index, brand_pct_sorted.values, color=colors, edgecolor="black")

# Add the percentage text to the end of each bar
for bar in bars:
    width = bar.get_width()
    # Place text slightly to the right of the bar
    ax.text(width + 0.5, bar.get_y() + bar.get_height()/2, 
            f'{width:.1f}%', 
            va='center', ha='left', fontsize=11, fontweight='bold')

# Emphasize the top brand's label on the Y-axis
ax.get_yticklabels()[list(brand_pct_sorted.index).index(top_brand)].set_fontweight('bold')
ax.get_yticklabels()[list(brand_pct_sorted.index).index(top_brand)].set_color('#FF4C4C')

# Formatting
ax.set_xlabel("Percentage of Laptops (%)", fontsize=12, fontweight='bold')
ax.set_ylabel("Laptop Brand", fontsize=12, fontweight='bold')
ax.set_title(f"Laptop Brand Distribution\n(Top brand: {top_brand} — {brand_pct.max():.1f}%)", fontsize=14, fontweight='bold', pad=15)

# Set x-axis limit to make room for the text labels
ax.set_xlim(0, max(brand_pct_sorted.values) + 5)

# Add a subtle grid for readability
ax.grid(axis='x', linestyle='--', alpha=0.7)
ax.set_axisbelow(True) # Put grid behind bars

plt.tight_layout()
plt.show()

In [ ]:
# Load and Clean Data
df = pd.read_csv("laptops_cleaned.csv")

# Clean string columns
df["laptop_brand"] = df["laptop_brand"].astype(str).str.strip().str.upper()
df["gpu_model"] = df["gpu_model"].astype(str).str.strip().str.upper()
df["ram"] = df["ram"].astype(str).str.extract(r'(\d+)').astype(float)
df["price"] = pd.to_numeric(df["price"], errors='coerce')
df["specs_score"] = pd.to_numeric(df["specs_score"], errors='coerce')
df["vram_gb"] = pd.to_numeric(df["vram_gb"], errors='coerce')

# Drop rows missing critical data for the plots
df_clean = df.dropna(subset=['price', 'specs_score', 'ram', 'vram_gb'])

# Setup Dark Mode Theme and Color Palette
plt.style.use('dark_background')
brand_colors = plt.cm.tab20(np.linspace(0, 1, len(df_clean['laptop_brand'].unique())))
brand_color_map = dict(zip(df_clean['laptop_brand'].unique(), brand_colors))

print("Data loaded and cleaned successfully. You can now run the plot cells.")

In [ ]:
import matplotlib.patches as mpatches
import math

fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(18, 20), facecolor='#121212')

# Flatten the 2D array of axes for easy iteration
axes = axes.flatten()

brands = df_clean['laptop_brand'].unique()

for i, brand in enumerate(brands):
    ax = axes[i]
    ax.set_facecolor('#1E1E1E')
    
    # Filter data for the specific brand
    subset = df_clean[df_clean['laptop_brand'] == brand]
    
    # Plot the scatter
    ax.scatter(subset['price'], subset['specs_score'], 
               s=subset['ram']*15, 
               alpha=0.8, 
               color=brand_color_map[brand], 
               edgecolors='white', linewidth=1)
    
    # Formatting for each mini-plot
    ax.set_title(brand, color='white', fontsize=14, fontweight='bold', pad=10)
    ax.tick_params(colors='#B0B0B0', labelsize=9)
    ax.grid(True, linestyle='--', alpha=0.2)
    
    # Format the X-axis
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f'₹{int(x/1000)}k'))
    ax.tick_params(axis='x', rotation=45) # Rotate x-labels to prevent overlap

    # Only add y-label to the leftmost plots to save space
    if i % 4 == 0:
        ax.set_ylabel('Specs Score', color='#B0B0B0', fontsize=10)
        
    # Only add x-label to the bottom row of plots
    if i >= len(brands) - 4:
        ax.set_xlabel('Price (₹)', color='#B0B0B0', fontsize=10)

# Turn off any empty subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

# Add a main title to the whole figure
fig.suptitle('Value Proposition: Price vs. Specs Score (By Brand)', 
             color="#FFFFFF", fontsize=22, fontweight='bold', y=0.98)

plt.tight_layout()
fig.subplots_adjust(top=0.92) # Make room for the main title
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8), facecolor='#121212')
ax.set_facecolor('#1E1E1E')

top_10 = df_clean.nlargest(10, 'price').sort_values('price', ascending=True)
# Shorten names for the Y-axis so they fit nicely
short_names = [name.split('(')[0].strip()[:40] + "..." for name in top_10['name']]
colors = [brand_color_map[b] for b in top_10['laptop_brand']]

bars = ax.barh(short_names, top_10['price'], color=colors, edgecolor='white', linewidth=1)

# Add price labels to the end of each bar
for bar in bars:
    width = bar.get_width()
    ax.text(width + 5000, bar.get_y() + bar.get_height()/2, 
            f'₹{int(width/1000)}k', va='center', ha='left', color='white', fontweight='bold')

ax.set_title('Top 10 Most Expensive Laptops', color='white', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Price (₹)', color='#B0B0B0', fontsize=12)
ax.tick_params(colors='#B0B0B0')
ax.grid(axis='x', linestyle='--', alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f'₹{int(x/1000)}k'))
ax.set_xlim(0, max(top_10['price']) + 100000) # Add space for text

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8), facecolor='#121212')
ax.set_facecolor('#1E1E1E')

# Get top 6 GPUs and group the rest as 'OTHER'
gpu_counts = df_clean['gpu_model'].value_counts().head(6)
other_count = df_clean['gpu_model'].value_counts().iloc[6:].sum()
if other_count > 0:
    gpu_counts['OTHER'] = other_count

# Explode the largest slice
explode = [0.1 if i == 0 else 0 for i in range(len(gpu_counts))]
cmap = plt.cm.plasma(np.linspace(0.2, 0.9, len(gpu_counts)))

wedges, texts, autotexts = ax.pie(gpu_counts, 
                                autopct='%1.1f%%', 
                                startangle=140, 
                                colors=cmap,
                                explode=explode,
                                pctdistance=0.85,
                                textprops=dict(color="w", weight="bold"))

# Draw a circle in the center to make it a donut
centre_circle = plt.Circle((0,0),0.70,fc='#1E1E1E')
fig.gca().add_artist(centre_circle)

ax.set_title('GPU Market Share', color='white', fontsize=16, fontweight='bold', pad=15)
ax.legend(wedges, gpu_counts.index, title="GPU Models", loc="center left",
        bbox_to_anchor=(1, 0, 0.5, 1), facecolor='#1E1E1E', edgecolor='none', labelcolor='white')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.ticker as ticker

fig, ax = plt.subplots(figsize=(12, 8), facecolor='#121212')
ax.set_facecolor('#1E1E1E')

# Add slight jitter to VRAM to prevent overlapping dots
jitter_vram = df_clean['vram_gb'] + np.random.normal(0, 0.1, size=len(df_clean))

scatter = ax.scatter(df_clean['price'], jitter_vram, 
                    c=df_clean['specs_score'], cmap='cool', 
                    s=100, alpha=0.8, edgecolors='white', linewidth=0.5)

ax.set_title('Price vs. VRAM (Colored by Specs Score)', color='white', fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Price (₹)', color='#B0B0B0', fontsize=12)
ax.set_ylabel('VRAM (GB)', color='#B0B0B0', fontsize=12)
ax.tick_params(colors='#B0B0B0')
ax.grid(True, linestyle='--', alpha=0.3)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, pos: f'₹{int(x/1000)}k'))

# FORCE Y-AXIS TO USE EVEN NUMBERS
# This sets the locator to only pick ticks in steps of 2, using integers.
ax.yaxis.set_major_locator(ticker.MultipleLocator(2))

# Add colorbar
cbar = fig.colorbar(scatter, ax=ax, pad=0.02)
cbar.set_label('Specs Score', color='#B0B0B0', fontsize=12)
cbar.ax.yaxis.set_tick_params(color='#B0B0B0')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='#B0B0B0')

plt.tight_layout()
plt.show()

In [ ]:
# the setup 
fig, ax = plt.subplots(figsize=(12, 8), facecolor='#121212')
ax.set_facecolor('#1E1E1E')

brands = df['laptop_brand'].unique()

for brand in brands:

    brand_data = df[df['laptop_brand'] == brand]

    ax.scatter(brand_data['price'],
            brand_data['specs_score'],
            label = brand,
            alpha = 0.3,
            s = 80)



ax.set_title("Laptop vs Spec Score")

ax.set_xlabel("Price in INR")
ax.set_ylabel("Spec Score")

ax.grid(True, linestyle='--', alpha=0.3)

ax.legend()
plt.show()